WTS: contrastive loss can be used to train Tversky Similarity directly (e.g. no external classification task for Tversky Projection) for query-ability.

(start with laptop distance as concept $c$)
1. make a dataset $D_c$ for a concept $c$ which contains two sets of trajectories (represented as embeddings $\phi$):
	* $P_c = \{\phi_p | \phi_p \text{ expresses concept } c\}$
	* $N_c = \{\phi_n | \phi_n \text{ does not express concept } c\}$
2. train a Tversky Similarity layer (learn $\Omega, \alpha, \beta,\theta$) with InfoNCE loss
3. inspect learned features, see if positive - negative is query-able

# 1. gridrobot dataset

In [ ]:
import numpy as np
data = np.load("../../../simulated_data/001-gridrobot/data/gridrobot_1960.npz")
all_trajs = data["trajs"]
all_feats = data["features"]

In [6]:
laptop_dist = all_feats[:,0]

In [10]:
laptop_max_indices = np.where(laptop_dist == laptop_dist.max())
laptop_min_indices = np.where(laptop_dist == laptop_dist.min())

In [15]:
hi_trajs = all_trajs[laptop_max_indices]
lo_trajs = all_trajs[laptop_min_indices]
print(f"{len(hi_trajs)} laptop max trajs and {len(lo_trajs)} laptop min trajs")

56 laptop max trajs and 448 laptop min trajs


# 2. TverskySimilarity layer with InfoNCE loss

In [ ]:
# import torch
import torch.nn as nn
# import torch.optim as optim
# from torch.optim.lr_scheduler import ExponentialLR
import sys
sys.path.insert(0, '../src')
from tversky import nn as tnn
from info_nce import InfoNCE, info_nce


class LiterallyJustTverskySim(nn.Module):
    def __init__(self, 
                 input_dim=19, 
                 fbank_size=4, 
                 similarity_model='contrast',
                 intersection_reduction='product',
                 difference_reduction='ignorematch',
                 normalize=False
                 ):
        super().__init__()
        self.tversky_sim = tnn.TverskySimilarity(
            embedding_dim=input_dim,
            fbank_size=fbank_size,
            similarity_model=similarity_model,
            normalize=normalize,
            intersection_reduction=intersection_reduction,
            difference_reduction=difference_reduction
        )
    
    def similarity(self, a, b):
        return self.tversky_sim(a,b)

In [33]:
import torch
import torch.nn.functional as F

TAU = 0.1
BATCH_SIZE = 16
EPOCHS = 10000
PRINT_EVERY = 100

model = LiterallyJustTverskySim()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

hi = torch.as_tensor(hi_trajs, dtype=torch.float32)   # (N_hi, d)
lo = torch.as_tensor(lo_trajs, dtype=torch.float32)   # (N_lo, d)

def contrastive_step(model, hi, lo, batch_size, tau=0.1):
    # anchors: batch_size positives; partners: also batch_size positives
    a_idx = torch.randperm(len(hi))[:batch_size]
    p_idx = torch.randperm(len(hi))[:batch_size]          # different draw -> different positive
    anchors  = hi[a_idx]                          # (batch_size, d)
    partners = hi[p_idx]                          # (batch_size, d)
    negs     = lo[torch.randperm(len(lo))[:batch_size]]    # (batch_size, d)  the negative pool

    # pairwise sim returns (rows, cols)
    s_pos = model.similarity(anchors, partners).diagonal().unsqueeze(1)  # (batch_size, 1): anchor i vs partner i
    s_neg = model.similarity(anchors, negs)                              # (batch_size, batch_size): anchor i vs every neg

    logits = torch.cat([s_pos, s_neg], dim=1) / tau   # (batch_size, 1+batch_size)
    labels = torch.zeros(batch_size, dtype=torch.long)          # positive is column 0
    return F.cross_entropy(logits, labels)

for epoch in range(EPOCHS):
    optimizer.zero_grad()
    loss = contrastive_step(model, hi, lo, BATCH_SIZE, TAU)
    loss.backward()
    optimizer.step()
    if epoch % PRINT_EVERY == 0:
        print(loss.item())

2043.593994140625
1186.6903076171875
1424.385009765625
876.6270751953125
718.3131713867188
462.7015686035156
239.63070678710938
66.79290771484375
135.61416625976562
107.66902160644531
81.4025650024414
59.48177719116211
41.62779235839844
27.233972549438477
35.84994888305664
16.154699325561523
15.720791816711426
13.411917686462402
22.77187728881836
13.30972957611084
10.834717750549316
12.290258407592773
9.064735412597656
8.669645309448242
6.046672344207764
8.449524879455566
4.471601486206055
3.281308650970459
3.248884677886963
4.491203784942627
3.3076326847076416
2.512070655822754
2.918666362762451
3.453005790710449
2.6754658222198486
2.799765110015869
3.544447898864746
3.417738437652588
2.6098148822784424
3.158019542694092
1.5056722164154053
2.4305291175842285
2.532808542251587
2.2644333839416504
1.6082379817962646
2.2258505821228027
2.4522464275360107
1.4331176280975342
1.9752085208892822
2.3432977199554443
2.0237293243408203
1.4779752492904663
2.013258457183838
2.088779926300049
1.893

# 3. inspect learned features, see if positive - negative is query-able

In [34]:
from tversky_utils import *

TODO does positive - negative, negative - positive lead to different t-test of means on true feature value?
TODO how consistent are positive - negative vs negative - positive feature sets?
TODO try for different feature bank sizes

In [ ]:
from scipy import stats

N_QUERIES = 10
accs = []
for i in range(N_QUERIES):
    hi, lo = train_pairs[i]
    hi_res = run_query(hi, lo) # hi - lo
    lo_res = run_query(lo, hi) # lo - hi
    hi_votes.update(hi_res["semantic_features"])
    lo_votes.update(lo_res["semantic_features"])

    thresh = VOTE_FRAC * (i + 1)
    hi_feats = {f for f, c in hi_votes.items() if c >= thresh}
    lo_feats = {f for f, c in lo_votes.items() if c >= thresh}

    hi_top_instances = get_top_instances(centered_trajs, feature_bank, hi_feats, TOP_RESULT_COUNT)
    lo_top_instances = get_top_instances(centered_trajs, feature_bank, lo_feats, TOP_RESULT_COUNT)

    # t test of means (laptop only)
    hi_top_instances_true_feats = [all_feats[inst["item_ix"], 0] for inst in hi_top_instances]
    lo_top_instances_true_feats = [all_feats[inst["item_ix"], 0] for inst in lo_top_instances]
    # both_constant = np.std(hi_top_instances) == 0 and np.std(lo_top_instances) == 0
    # if len(a) >= 2 and len(b) >= 2 and not both_constant:
    t = stats.ttest_ind(hi_top_instances_true_feats, lo_top_instances_true_feats, alternative="greater")
    print(f"ttest pvalue after {i + 1} queries: {t.pvalue}  "
          f"(|hi_feats|={len(hi_feats)}, |lo_feats|={len(lo_feats)})")